# Vortex Panel Methods in 2D
### Constant-strength and linear-strength vortex panel methods: implementation, verification and limitations

**Author:** *(your name, roll number)*   **Course:** *(course code)*   **Submitted:** 8 November 2026

---

**Contents**
1. Algorithms
2. Implementation in Python
3. Verification on a circular cylinder
4. Verification on symmetric and cambered airfoils
5. Prediction of airfoil characteristics
6. Limitations of prediction: effect of thickness
7. References and acknowledgements

**Notation** (Anderson, *Fundamentals of Aerodynamics*, Sec. 4.10)

| Symbol | Meaning |
|---|---|
| $(X_j, Y_j)$ | boundary points (panel end points), clockwise from the trailing edge, lower surface first |
| $(x_i, y_i)$ | control points (panel midpoints) |
| $S_j,\ \phi_j$ | length and angle of panel $j$ |
| $\beta_i = \phi_i + \pi/2 - \alpha$ | angle between $V_\infty$ and the outward normal $n_i$ |
| $\gamma$ | vortex strength per unit length (clockwise positive) |
| $\theta_{ij}$ | angle $\tan^{-1}[(y_i - y_j)/(x_i - x_j)]$ from a point on panel $j$ to control point $i$ |
| $J_{ij} = \int_j \partial\theta_{ij}/\partial n_i\, ds_j$ | normal influence of panel $j$ at control point $i$ |
| $K_{ij} = \int_j \partial\theta_{ij}/\partial s_i\, ds_j$ | tangential influence of panel $j$ at control point $i$ |
| $J'_{ij} = \int_j (s_j/S_j)\,\partial\theta_{ij}/\partial n_i\, ds_j$ | weighted normal influence (linear panels) |

## 1. Algorithms

The airfoil surface is replaced by $N$ straight panels joined at $N+1$ boundary points. Each panel carries a vortex sheet. The sheet strengths are chosen so that

* the surface is a streamline: the normal velocity is zero at each control point, and
* the Kutta condition holds: the flow leaves the trailing edge smoothly, $\gamma(\text{TE}) = 0$.

The normal velocity induced at control point $i$ by a vortex sheet on panel $j$ is $-\frac{1}{2\pi}\int_j \gamma(s_j)\, \frac{\partial \theta_{ij}}{\partial n_i}\, ds_j$, and the freestream contributes $V_\infty \cos\beta_i$. Differentiating $\theta_{ij}$ gives the integrands

$$\frac{\partial\theta_{ij}}{\partial n_i} = \frac{(x_i - x_j)\cos\phi_i + (y_i - y_j)\sin\phi_i}{(x_i-x_j)^2 + (y_i-y_j)^2}, \qquad
\frac{\partial\theta_{ij}}{\partial s_i} = \frac{(x_i - x_j)\sin\phi_i - (y_i - y_j)\cos\phi_i}{(x_i-x_j)^2 + (y_i-y_j)^2},$$

where $(x_j, y_j) = (X_j + s_j\cos\phi_j,\; Y_j + s_j\sin\phi_j)$ moves along panel $j$.

### Algorithm 1: constant-strength vortex panel method (`vortex_panel_method_0`)

**Input:** boundary points $(X_j, Y_j)$, $j = 1,\dots,N+1$ (clockwise, closed); $\alpha$; $V_\infty$.
**Output:** $\gamma_j$, $V_i$, $C_{p,i}$, $\Gamma$, $c_l$.

1. **Geometry.** For each panel $j = 1,\dots,N$: $S_j = \sqrt{(X_{j+1}-X_j)^2 + (Y_{j+1}-Y_j)^2}$, $\phi_j = \operatorname{atan2}(Y_{j+1}-Y_j,\, X_{j+1}-X_j)$, control point $(x_j, y_j)$ at the panel midpoint, and $\beta_j = \phi_j + \pi/2 - \alpha$.
2. **Influence coefficients.** For every pair $i \ne j$, evaluate $J_{ij}$ and $K_{ij}$ by numerical quadrature. For $i = j$ the principal-value integrals vanish: $J_{ii} = K_{ii} = 0$.
3. **Flow tangency.** For $i = 1,\dots,N$: $\displaystyle\sum_{j=1}^{N} J_{ij}\,\gamma_j = 2\pi V_\infty\cos\beta_i$.
4. **Kutta condition.** $\gamma_1 + \gamma_N = 0$. This gives $N+1$ equations for $N$ unknowns, so the last tangency equation is discarded and replaced by the Kutta condition (Anderson's procedure).
5. **Solve** the $N\times N$ system for $\gamma_j$. *Remedy for ill-conditioning (Section 4.2):* the matrix has a nearly singular "sawtooth" mode $\gamma = (+a, -a, +a, \dots)$. The system is solved by singular value decomposition, $A = U\Sigma V^T$, keeping every mode except the one with the smallest singular value.
6. **Surface velocity.** $\displaystyle V_i = V_\infty\sin\beta_i - \sum_{j} \frac{\gamma_j}{2\pi} K_{ij} + \frac{\gamma_i}{2}$, where $\gamma_i/2$ is the outer-side jump across the sheet. Then $C_{p,i} = 1 - (V_i/V_\infty)^2$.
7. **Lift.** $\Gamma = \sum_j \gamma_j S_j$ and $c_l = 2\Gamma/(V_\infty c)$ (Kutta–Joukowski).

*Non-lifting closed body (cylinder):* step 4 is replaced by zero circulation, $\sum_j \gamma_j S_j = 0$. This row is appended to the $N$ tangency equations and the $(N+1)\times N$ system is solved by least squares. Simply replacing an equation fails for even $N$ because $J$ then has two exactly singular modes (uniform and sawtooth).

### Algorithm 2: linear-strength vortex panel method (`vortex_panel_method_1`)

**Input:** as Algorithm 1. **Output:** nodal $\gamma_1,\dots,\gamma_{N+1}$, $C_p$ at the boundary points, $\Gamma$, $c_l$.

1. **Geometry.** As in Algorithm 1.
2. **Strength distribution.** On panel $j$: $\gamma(s_j) = \gamma_j\,(1 - s_j/S_j) + \gamma_{j+1}\, s_j/S_j$. $\gamma$ is continuous around the body and the unknowns are the $N+1$ nodal values.
3. **Influence coefficients.** For $i \ne j$, evaluate $J_{ij}$ and $J'_{ij}$. Panel $j$ contributes $(J_{ij} - J'_{ij})$ to column $j$ and $J'_{ij}$ to column $j+1$ of an $N \times (N+1)$ matrix $\mathbf{J}^{(1)}$.
   *Self-influence:* the principal value of the linear part of the sheet gives $\mathbf{J}^{(1)}_{i,i} \mathrel{+}= 1$, $\mathbf{J}^{(1)}_{i,i+1} \mathrel{+}= -1$.
4. **Flow tangency.** For $i = 1,\dots,N$: $\displaystyle\sum_{k=1}^{N+1} \mathbf{J}^{(1)}_{ik}\,\gamma_k = 2\pi V_\infty\cos\beta_i$.
5. **Kutta condition** as an additional, $(N+1)$-th equation: $\gamma_1 + \gamma_{N+1} = 0$. No tangency equation is discarded, and the system is square and well conditioned. (Cylinder: $\sum_j S_j(\gamma_j + \gamma_{j+1})/2 = 0$ instead.)
6. **Solve** the $(N+1)\times(N+1)$ system.
7. **Surface velocity.** Inside the body the fluid is at rest, so the surface speed equals the local sheet strength: $V_k = \gamma_k$ and $C_{p,k} = 1 - (\gamma_k/V_\infty)^2$ at the boundary points.
8. **Lift.** $\Gamma = \sum_j S_j(\gamma_j + \gamma_{j+1})/2$ (exact for linear $\gamma$), $c_l = 2\Gamma/(V_\infty c)$.

## 2. Implementation in Python

The code follows the two algorithms step by step. All functions take the boundary points and the angle of attack as arguments and return their results in a dictionary.

| Function | Purpose |
|---|---|
| `panel_geometry` | panel lengths, angles and control points |
| `naca4_boundary_points` | NACA 4-digit coordinates, cosine or uniform spacing, closed trailing edge |
| `compute_JK` | $J_{ij}$, $K_{ij}$ by adaptive Gauss–Kronrod quadrature (`scipy.integrate.quad`) |
| `vortex_panel_method_0` | Algorithm 1 |
| `compute_J_linear`, `vortex_panel_method_1` | Algorithm 2 |
| `forces_from_Cp` | $c_l$, $c_d$, $c_{m,c/4}$ by pressure integration |

### 2.1 Imports

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
import time

### 2.2 Geometry

In [2]:
def panel_geometry(XB, YB):
    """Panels from boundary points XB, YB (N+1,), clockwise, closed.
    Returns control points XC, YC, panel lengths S and panel angles phi (N,)."""
    dX, dY = np.diff(XB), np.diff(YB)
    S   = np.hypot(dX, dY)
    phi = np.mod(np.arctan2(dY, dX), 2*np.pi)
    XC  = 0.5*(XB[:-1] + XB[1:])
    YC  = 0.5*(YB[:-1] + YB[1:])
    return XC, YC, S, phi


def beta_angle(phi, alpha):
    """Angle between the freestream and the outward normal (alpha in radians)."""
    return phi + np.pi/2 - alpha


def cylinder_boundary_points(N, R=1.0):
    """N panels on a circle of radius R, clockwise from (R, 0)."""
    ang = np.linspace(0, -2*np.pi, N + 1)
    return R*np.cos(ang), R*np.sin(ang)


def naca4_boundary_points(code, N, spacing='cosine'):
    """NACA 4-digit airfoil, chord = 1, N panels (even).
    spacing: 'cosine' or 'uniform'.
    Returns XB, YB (N+1,), clockwise from the trailing edge, lower surface first."""
    m = int(code[0])/100
    p = int(code[1])/10
    t = int(code[2:])/100
    n = N//2

    if spacing == 'cosine':
        x = 0.5*(1 - np.cos(np.linspace(0, np.pi, n + 1)))
    else:
        x = np.linspace(0, 1, n + 1)

    yt = 5*t*(0.2969*np.sqrt(x) - 0.1260*x - 0.3516*x**2 + 0.2843*x**3 - 0.1036*x**4)

    if m == 0:
        yc = np.zeros_like(x); dyc = np.zeros_like(x)
    else:
        yc  = np.where(x < p, m/p**2*(2*p*x - x**2),
                              m/(1-p)**2*((1 - 2*p) + 2*p*x - x**2))
        dyc = np.where(x < p, 2*m/p**2*(p - x),
                              2*m/(1-p)**2*(p - x))
    th = np.arctan(dyc)

    xu = x - yt*np.sin(th); yu = yc + yt*np.cos(th)
    xl = x + yt*np.sin(th); yl = yc - yt*np.cos(th)

    XB = np.concatenate([xl[::-1], xu[1:]])
    YB = np.concatenate([yl[::-1], yu[1:]])
    return XB, YB